# EDA - All Datasets + Smartbuilding

This Kaggle notebook runs the exploratory data analysis pipeline for the benchmark datasets plus the cleaned `smartbuilding/smart.csv` dataset.

It does the following:
- installs dependencies
- clones your repo
- syncs the latest local `EDA_all_datasets_with_smart.py`
- downloads and extracts the dataset zip
- detects and previews `smart.csv`
- runs the EDA script
- saves summary CSVs, column reference files, and plots
- zips the full EDA output folder


In [ ]:
!pip install gdown seaborn --quiet
import os
import json
import shutil
import pandas as pd
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


In [ ]:
REPO_URL = "https://github.com/agamswami/SimpleTMG.git"
REPO_DIR = "/kaggle/working/SimpleTM"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")


In [ ]:
EDA_SCRIPT = "#!/usr/bin/env python\n\"\"\"\nEDA for the SimpleTM benchmark datasets plus the cleaned smartbuilding dataset.\n\nThe smartbuilding file is the cleaned hourly Floor-6 CU-BEMS data derived by\n`cleancode.py`. The original CU-BEMS documentation is in `smart.pdf`.\n\nThis script:\n1. discovers benchmark datasets under a dataset root\n2. includes `smartbuilding/smart.csv` when present\n3. saves a column reference and dataset summary\n4. generates lightweight per-dataset EDA plots\n5. adds smartbuilding-specific plots for load, temperature, lux, and zone totals\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport math\nfrom pathlib import Path\nfrom typing import Dict, Iterable, List, Optional, Tuple\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\n\ntry:\n    import seaborn as sns\nexcept ImportError:\n    sns = None\n\n\nplt.style.use(\"seaborn-v0_8-whitegrid\")\nif sns is not None:\n    sns.set_palette(\"tab10\")\n\nDATE_CANDIDATES = {\"date\", \"datetime\", \"timestamp\", \"time\"}\nEPS = 1e-8\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=\"EDA for all datasets including smartbuilding\")\n    parser.add_argument(\n        \"--dataset-root\",\n        type=str,\n        default=\"./dataset/SimpleTM_datasets\",\n        help=\"Root folder that contains ETT-small, weather, electricity, traffic, Solar, PEMS, smartbuilding\",\n    )\n    parser.add_argument(\n        \"--smart-csv\",\n        type=str,\n        default=\"./smart.csv\",\n        help=\"Fallback path for the cleaned smartbuilding CSV when it is not yet inside the dataset zip\",\n    )\n    parser.add_argument(\n        \"--output-dir\",\n        type=str,\n        default=\"./results_complete/eda_all_datasets_with_smart\",\n        help=\"Directory where summaries and plots will be saved\",\n    )\n    parser.add_argument(\n        \"--max-points\",\n        type=int,\n        default=2000,\n        help=\"Maximum number of points to use in single-series plots\",\n    )\n    return parser.parse_args()\n\n\ndef resolve_existing_path(candidates: Iterable[Path]) -> Optional[Path]:\n    for candidate in candidates:\n        if candidate.exists():\n            return candidate\n    return None\n\n\ndef build_dataset_specs(dataset_root: Path, smart_csv: Path) -> List[Dict[str, object]]:\n    specs = [\n        {\n            \"name\": \"ETTh1\",\n            \"kind\": \"csv\",\n            \"path\": resolve_existing_path([dataset_root / \"ETT-small\" / \"ETTh1.csv\"]),\n        },\n        {\n            \"name\": \"ETTh2\",\n            \"kind\": \"csv\",\n            \"path\": resolve_existing_path([dataset_root / \"ETT-small\" / \"ETTh2.csv\"]),\n        },\n        {\n            \"name\": \"ETTm1\",\n            \"kind\": \"csv\",\n            \"path\": resolve_existing_path([dataset_root / \"ETT-small\" / \"ETTm1.csv\"]),\n        },\n        {\n            \"name\": \"ETTm2\",\n            \"kind\": \"csv\",\n            \"path\": resolve_existing_path([dataset_root / \"ETT-small\" / \"ETTm2.csv\"]),\n        },\n        {\n            \"name\": \"weather\",\n            \"kind\": \"csv\",\n            \"path\": resolve_existing_path([dataset_root / \"weather\" / \"weather.csv\"]),\n        },\n        {\n            \"name\": \"electricity\",\n            \"kind\": \"csv\",\n            \"path\": resolve_existing_path([dataset_root / \"electricity\" / \"electricity.csv\"]),\n        },\n        {\n            \"name\": \"traffic\",\n            \"kind\": \"csv\",\n            \"path\": resolve_existing_path([dataset_root / \"traffic\" / \"traffic.csv\"]),\n        },\n        {\n            \"name\": \"Solar\",\n            \"kind\": \"solar_txt\",\n            \"path\": resolve_existing_path([dataset_root / \"Solar\" / \"solar_AL.txt\"]),\n        },\n        {\n            \"name\": \"PEMS03\",\n            \"kind\": \"pems_npz\",\n            \"path\": resolve_existing_path([dataset_root / \"PEMS\" / \"PEMS03.npz\"]),\n        },\n        {\n            \"name\": \"PEMS04\",\n            \"kind\": \"pems_npz\",\n            \"path\": resolve_existing_path([dataset_root / \"PEMS\" / \"PEMS04.npz\"]),\n        },\n        {\n            \"name\": \"PEMS07\",\n            \"kind\": \"pems_npz\",\n            \"path\": resolve_existing_path([dataset_root / \"PEMS\" / \"PEMS07.npz\"]),\n        },\n        {\n            \"name\": \"PEMS08\",\n            \"kind\": \"pems_npz\",\n            \"path\": resolve_existing_path([dataset_root / \"PEMS\" / \"PEMS08.npz\"]),\n        },\n        {\n            \"name\": \"smartbuilding\",\n            \"kind\": \"csv\",\n            \"path\": resolve_existing_path(\n                [dataset_root / \"smartbuilding\" / \"smart.csv\", smart_csv]\n            ),\n        },\n    ]\n    return [spec for spec in specs if spec[\"path\"] is not None]\n\n\ndef find_date_column(columns: Iterable[str]) -> Optional[str]:\n    for col in columns:\n        if col.lower() in DATE_CANDIDATES:\n            return col\n    return None\n\n\ndef load_dataset(spec: Dict[str, object]) -> Tuple[pd.DataFrame, Dict[str, object]]:\n    path = Path(spec[\"path\"])\n    kind = spec[\"kind\"]\n    name = str(spec[\"name\"])\n\n    if kind == \"csv\":\n        raw = pd.read_csv(path)\n        date_col = find_date_column(raw.columns)\n        if date_col is not None:\n            raw[date_col] = pd.to_datetime(raw[date_col], errors=\"coerce\")\n            raw = raw.dropna(subset=[date_col]).set_index(date_col)\n        numeric = raw.select_dtypes(include=[np.number]).copy()\n    elif kind == \"solar_txt\":\n        raw = pd.read_csv(path, header=None)\n        raw.columns = [f\"feature_{i:03d}\" for i in range(raw.shape[1])]\n        numeric = raw\n    elif kind == \"pems_npz\":\n        data = np.load(path, allow_pickle=True)[\"data\"][:, :, 0]\n        numeric = pd.DataFrame(data, columns=[f\"node_{i:03d}\" for i in range(data.shape[1])])\n    else:\n        raise ValueError(f\"Unsupported dataset kind: {kind}\")\n\n    numeric = numeric.replace([np.inf, -np.inf], np.nan)\n    metadata = {\n        \"name\": name,\n        \"path\": str(path),\n        \"kind\": kind,\n        \"has_datetime_index\": isinstance(numeric.index, pd.DatetimeIndex),\n    }\n    return numeric, metadata\n\n\ndef infer_frequency(index: pd.Index) -> str:\n    if not isinstance(index, pd.DatetimeIndex) or len(index) < 4:\n        return \"\"\n    try:\n        freq = pd.infer_freq(index[: min(len(index), 200)])\n        return freq or \"\"\n    except ValueError:\n        return \"\"\n\n\ndef classify_column(name: str) -> str:\n    lower = name.lower()\n    if \"total(kw)\" in lower:\n        return \"zone_or_floor_total_kw\"\n    if \"(kw)\" in lower:\n        return \"power_kw\"\n    if \"degc\" in lower or \"temp\" in lower:\n        return \"temperature\"\n    if \"rh%\" in lower or \"humidity\" in lower:\n        return \"humidity\"\n    if \"lux\" in lower:\n        return \"light\"\n    return \"other\"\n\n\ndef spectral_entropy(signal: np.ndarray) -> float:\n    signal = signal[np.isfinite(signal)]\n    if signal.size < 4:\n        return float(\"nan\")\n    centered = signal - np.mean(signal)\n    power = np.abs(np.fft.rfft(centered)) ** 2\n    power = power[1:]\n    if power.size == 0 or np.sum(power) <= EPS:\n        return 0.0\n    probs = power / np.sum(power)\n    entropy = -np.sum(probs * np.log(probs + EPS))\n    return float(entropy / np.log(len(probs) + EPS))\n\n\ndef dominant_period(signal: np.ndarray) -> float:\n    signal = signal[np.isfinite(signal)]\n    if signal.size < 4:\n        return float(\"nan\")\n    centered = signal - np.mean(signal)\n    power = np.abs(np.fft.rfft(centered)) ** 2\n    freq = np.fft.rfftfreq(len(centered), d=1.0)\n    if len(power) <= 1:\n        return float(\"nan\")\n    power[0] = 0.0\n    best = int(np.argmax(power))\n    if best <= 0 or freq[best] <= 0:\n        return float(\"nan\")\n    return float(1.0 / freq[best])\n\n\ndef safe_series(df: pd.DataFrame) -> pd.Series:\n    if \"Floor_Total(kW)\" in df.columns:\n        return df[\"Floor_Total(kW)\"]\n    return df.iloc[:, 0]\n\n\ndef save_column_reference(\n    dataset_name: str,\n    df: pd.DataFrame,\n    output_dir: Path,\n    rows: List[Dict[str, object]],\n) -> None:\n    lines = [f\"Dataset: {dataset_name}\", f\"Columns: {len(df.columns)}\", \"\"]\n    for idx, col in enumerate(df.columns, start=1):\n        role = classify_column(col)\n        dtype = str(df[col].dtype)\n        lines.append(f\"{idx:03d}. {col} | dtype={dtype} | role={role}\")\n        rows.append(\n            {\n                \"Dataset\": dataset_name,\n                \"Column\": col,\n                \"DType\": dtype,\n                \"Role\": role,\n            }\n        )\n    (output_dir / \"columns.txt\").write_text(\"\\n\".join(lines), encoding=\"utf-8\")\n\n\ndef plot_representative_series(df: pd.DataFrame, output_dir: Path, dataset_name: str, max_points: int) -> None:\n    series = safe_series(df).iloc[:max_points]\n    plt.figure(figsize=(13, 4))\n    plt.plot(series.index if isinstance(series.index, pd.DatetimeIndex) else np.arange(len(series)), series.values, linewidth=1.0)\n    plt.title(f\"{dataset_name} - Representative Series ({series.name})\")\n    plt.ylabel(series.name)\n    plt.tight_layout()\n    plt.savefig(output_dir / \"representative_series.png\", dpi=180)\n    plt.close()\n\n\ndef plot_corr_heatmap(df: pd.DataFrame, output_dir: Path, dataset_name: str) -> None:\n    subset = df.copy()\n    if subset.shape[1] > 12:\n        top_cols = subset.var().sort_values(ascending=False).head(12).index\n        subset = subset[top_cols]\n    corr = subset.corr()\n    plt.figure(figsize=(10, 8))\n    if sns is not None:\n        sns.heatmap(corr, cmap=\"coolwarm\", center=0, square=False)\n    else:\n        plt.imshow(corr.values, cmap=\"coolwarm\", aspect=\"auto\", vmin=-1, vmax=1)\n        plt.colorbar()\n        plt.xticks(range(len(corr.columns)), corr.columns, rotation=90, fontsize=8)\n        plt.yticks(range(len(corr.index)), corr.index, fontsize=8)\n    plt.title(f\"{dataset_name} - Correlation Heatmap\")\n    plt.tight_layout()\n    plt.savefig(output_dir / \"correlation_heatmap.png\", dpi=180)\n    plt.close()\n\n\ndef plot_daily_profile(df: pd.DataFrame, output_dir: Path, dataset_name: str) -> None:\n    if not isinstance(df.index, pd.DatetimeIndex):\n        return\n    series = safe_series(df)\n    hourly = series.groupby(df.index.hour).mean()\n    plt.figure(figsize=(10, 4))\n    plt.plot(hourly.index, hourly.values, marker=\"o\")\n    plt.title(f\"{dataset_name} - Mean Hour-of-Day Profile\")\n    plt.xlabel(\"Hour\")\n    plt.ylabel(series.name)\n    plt.tight_layout()\n    plt.savefig(output_dir / \"daily_profile.png\", dpi=180)\n    plt.close()\n\n\ndef plot_power_spectrum(df: pd.DataFrame, output_dir: Path, dataset_name: str) -> None:\n    series = safe_series(df).dropna().values\n    if len(series) < 8:\n        return\n    centered = series - series.mean()\n    fft_vals = np.abs(np.fft.rfft(centered)) ** 2\n    freqs = np.fft.rfftfreq(len(centered), d=1.0)\n    upper = max(8, len(freqs) // 4)\n    plt.figure(figsize=(10, 4))\n    plt.semilogy(freqs[1:upper], fft_vals[1:upper] + EPS)\n    plt.title(f\"{dataset_name} - Power Spectrum\")\n    plt.xlabel(\"Frequency\")\n    plt.ylabel(\"Power\")\n    plt.tight_layout()\n    plt.savefig(output_dir / \"power_spectrum.png\", dpi=180)\n    plt.close()\n\n\ndef plot_smartbuilding_specific(df: pd.DataFrame, output_dir: Path) -> None:\n    zone_total_cols = [\n        col for col in df.columns\n        if col.endswith(\"_Total(kW)\") and col != \"Floor_Total(kW)\"\n    ]\n\n    summary_rows = []\n    for label, matcher in [\n        (\"power_kw\", lambda c: \"(kW)\" in c and \"Total\" not in c),\n        (\"zone_total_kw\", lambda c: c.endswith(\"_Total(kW)\") and c != \"Floor_Total(kW)\"),\n        (\"temperature\", lambda c: \"degC\" in c or \"Temp\" in c),\n        (\"humidity\", lambda c: \"RH%\" in c or \"humidity\" in c.lower()),\n        (\"lux\", lambda c: \"lux\" in c.lower()),\n    ]:\n        cols = [col for col in df.columns if matcher(col)]\n        summary_rows.append({\"Group\": label, \"Count\": len(cols), \"Columns\": \", \".join(cols)})\n\n    pd.DataFrame(summary_rows).to_csv(output_dir / \"smartbuilding_column_groups.csv\", index=False)\n\n    if zone_total_cols:\n        zoom = df[zone_total_cols + [\"Floor_Total(kW)\"]].iloc[: min(len(df), 24 * 14)]\n        plt.figure(figsize=(13, 5))\n        for col in zone_total_cols:\n            plt.plot(zoom.index, zoom[col], alpha=0.75, linewidth=1.0, label=col)\n        plt.plot(zoom.index, zoom[\"Floor_Total(kW)\"], color=\"black\", linewidth=2.0, label=\"Floor_Total(kW)\")\n        plt.title(\"smartbuilding - Zone Totals and Floor Total\")\n        plt.ylabel(\"kW\")\n        plt.legend(loc=\"upper right\", fontsize=8, ncol=2)\n        plt.tight_layout()\n        plt.savefig(output_dir / \"smartbuilding_zone_totals.png\", dpi=180)\n        plt.close()\n\n    if {\"Floor_Total(kW)\", \"Floor_Mean_Temp\"}.issubset(df.columns):\n        plt.figure(figsize=(7, 5))\n        plt.scatter(df[\"Floor_Mean_Temp\"], df[\"Floor_Total(kW)\"], alpha=0.35, s=8)\n        plt.title(\"smartbuilding - Floor Total vs Mean Temperature\")\n        plt.xlabel(\"Floor_Mean_Temp\")\n        plt.ylabel(\"Floor_Total(kW)\")\n        plt.tight_layout()\n        plt.savefig(output_dir / \"smartbuilding_temp_vs_energy.png\", dpi=180)\n        plt.close()\n\n    if isinstance(df.index, pd.DatetimeIndex) and \"Floor_Total(kW)\" in df.columns:\n        hourly_heatmap = (\n            df.assign(hour=df.index.hour, weekday=df.index.day_name())\n            .pivot_table(index=\"weekday\", columns=\"hour\", values=\"Floor_Total(kW)\", aggfunc=\"mean\")\n            .reindex([\"Monday\", \"Tuesday\", \"Wednesday\", \"Thursday\", \"Friday\", \"Saturday\", \"Sunday\"])\n        )\n        plt.figure(figsize=(12, 4))\n        if sns is not None:\n            sns.heatmap(hourly_heatmap, cmap=\"viridis\")\n        else:\n            plt.imshow(hourly_heatmap.values, cmap=\"viridis\", aspect=\"auto\")\n            plt.colorbar()\n            plt.xticks(range(hourly_heatmap.shape[1]), hourly_heatmap.columns, fontsize=8)\n            plt.yticks(range(hourly_heatmap.shape[0]), hourly_heatmap.index, fontsize=8)\n        plt.title(\"smartbuilding - Mean Floor Total by Weekday and Hour\")\n        plt.tight_layout()\n        plt.savefig(output_dir / \"smartbuilding_weekday_hour_heatmap.png\", dpi=180)\n        plt.close()\n\n\ndef main() -> None:\n    args = parse_args()\n    dataset_root = Path(args.dataset_root)\n    smart_csv = Path(args.smart_csv)\n    output_root = Path(args.output_dir)\n    output_root.mkdir(parents=True, exist_ok=True)\n\n    dataset_specs = build_dataset_specs(dataset_root, smart_csv)\n    if not dataset_specs:\n        raise FileNotFoundError(\n            \"No datasets were found. Check --dataset-root and --smart-csv.\"\n        )\n\n    summary_rows: List[Dict[str, object]] = []\n    column_rows: List[Dict[str, object]] = []\n\n    for spec in dataset_specs:\n        df, metadata = load_dataset(spec)\n        dataset_name = str(spec[\"name\"])\n        dataset_dir = output_root / dataset_name\n        dataset_dir.mkdir(parents=True, exist_ok=True)\n\n        save_column_reference(dataset_name, df, dataset_dir, column_rows)\n        df.describe().transpose().to_csv(dataset_dir / \"describe.csv\")\n\n        missing_total = int(df.isna().sum().sum())\n        zero_ratio = float((df == 0).sum().sum() / max(df.size, 1))\n        series = safe_series(df).dropna().values\n\n        summary_rows.append(\n            {\n                \"Dataset\": dataset_name,\n                \"Kind\": metadata[\"kind\"],\n                \"Path\": metadata[\"path\"],\n                \"Rows\": len(df),\n                \"NumericColumns\": df.shape[1],\n                \"MissingValues\": missing_total,\n                \"ZeroRatio\": zero_ratio,\n                \"Start\": df.index.min() if isinstance(df.index, pd.DatetimeIndex) else \"\",\n                \"End\": df.index.max() if isinstance(df.index, pd.DatetimeIndex) else \"\",\n                \"InferredFreq\": infer_frequency(df.index),\n                \"RepresentativeSeries\": safe_series(df).name,\n                \"SpectralEntropy\": spectral_entropy(series),\n                \"DominantPeriod\": dominant_period(series),\n            }\n        )\n\n        plot_representative_series(df, dataset_dir, dataset_name, args.max_points)\n        plot_corr_heatmap(df, dataset_dir, dataset_name)\n        plot_daily_profile(df, dataset_dir, dataset_name)\n        plot_power_spectrum(df, dataset_dir, dataset_name)\n\n        if dataset_name == \"smartbuilding\":\n            plot_smartbuilding_specific(df, dataset_dir)\n\n        print(f\"Saved EDA outputs for {dataset_name} -> {dataset_dir}\")\n\n    summary_df = pd.DataFrame(summary_rows).sort_values(\"Dataset\").reset_index(drop=True)\n    columns_df = pd.DataFrame(column_rows).sort_values([\"Dataset\", \"Column\"]).reset_index(drop=True)\n\n    summary_df.to_csv(output_root / \"dataset_summary.csv\", index=False)\n    columns_df.to_csv(output_root / \"column_reference.csv\", index=False)\n\n    print(\"\\nDataset summary:\")\n    print(summary_df.to_string(index=False))\n    print(f\"\\nSummary saved to {output_root / 'dataset_summary.csv'}\")\n    print(f\"Column reference saved to {output_root / 'column_reference.csv'}\")\n\n\nif __name__ == \"__main__\":\n    main()\n"

with open(os.path.join(REPO_DIR, 'EDA_all_datasets_with_smart.py'), 'w', encoding='utf-8', newline='') as f:
    f.write(EDA_SCRIPT)

print('Synced EDA_all_datasets_with_smart.py')


In [ ]:
FILE_ID = "1hTpUrhe1yEIGa9mCiGxM5rDyzlYKAnyx"
OUTPUT_PATH = "/kaggle/working/dataset.zip"
DATASET_DIR = "./dataset"

if not os.path.exists(DATASET_DIR):
    !gdown https://drive.google.com/uc?id={FILE_ID} -O {OUTPUT_PATH}
    import zipfile
    os.makedirs(DATASET_DIR, exist_ok=True)
    with zipfile.ZipFile(OUTPUT_PATH, 'r') as zip_ref:
        zip_ref.extractall(DATASET_DIR)
    print("Dataset extracted successfully.")
else:
    print("Dataset already exists.")

for root, dirs, files in os.walk(DATASET_DIR):
    level = root.replace(DATASET_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files[:10]:
        print(f'{subindent}{file}')


In [ ]:
smart_candidates = [
    ("./dataset/SimpleTM_datasets/smartbuilding", "smart.csv"),
    ("./dataset/SimpleTM_datasets/Smartbuilding", "smart.csv"),
    (".", "smart.csv"),
]

SMART_ROOT, SMART_PATH = None, None
for root, path in smart_candidates:
    candidate = os.path.join(root, path)
    if os.path.exists(candidate):
        SMART_ROOT, SMART_PATH = root, path
        break

if SMART_ROOT is None:
    raise FileNotFoundError("smart.csv not found in the extracted dataset or repo root.")

smart_csv_path = os.path.join(SMART_ROOT, SMART_PATH)
smart_df = pd.read_csv(smart_csv_path)
print("smart.csv path:", smart_csv_path)
print("smart.csv shape:", smart_df.shape)
print("smart.csv columns:")
for i, col in enumerate(smart_df.columns, start=1):
    print(f"{i:02d}. {col}")
print()
print(smart_df.head(3).to_string())


In [ ]:
EDA_OUTPUT_DIR = "/kaggle/working/eda_all_datasets_with_smart"

cmd = [
    "python", "EDA_all_datasets_with_smart.py",
    "--dataset-root", "./dataset/SimpleTM_datasets",
    "--smart-csv", smart_csv_path,
    "--output-dir", EDA_OUTPUT_DIR,
]

import subprocess
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
for line in process.stdout:
    print(line, end="")
process.wait()

if process.returncode != 0:
    raise RuntimeError(f"EDA script failed with exit code {process.returncode}")


In [ ]:
summary_path = os.path.join(EDA_OUTPUT_DIR, 'dataset_summary.csv')
column_ref_path = os.path.join(EDA_OUTPUT_DIR, 'column_reference.csv')
smart_groups_path = os.path.join(EDA_OUTPUT_DIR, 'smartbuilding', 'smartbuilding_column_groups.csv')

if os.path.exists(summary_path):
    summary_df = pd.read_csv(summary_path)
    print('DATASET SUMMARY')
    print(summary_df.to_string(index=False))

if os.path.exists(column_ref_path):
    cols_df = pd.read_csv(column_ref_path)
    print()
    print('COLUMN REFERENCE PREVIEW')
    print(cols_df.head(20).to_string(index=False))

if os.path.exists(smart_groups_path):
    smart_groups_df = pd.read_csv(smart_groups_path)
    print()
    print('SMARTBUILDING COLUMN GROUPS')
    print(smart_groups_df.to_string(index=False))


In [ ]:
print('Packing EDA outputs...')
archive_path = shutil.make_archive('/kaggle/working/EDA_All_Datasets_With_Smart', 'zip', EDA_OUTPUT_DIR)
print(f'\nDONE! Download {archive_path} from the Kaggle Output pane.')
